# Hasse *et al.* PRA **109**, 053105 (2024) — Numerics Reproduction

Stroboscopic phase-space probing of a single trapped $^{25}\mathrm{Mg}^{+}$ ion. This notebook is a **self-contained reproduction of the simulated figures** (Figs. 3, 6, 8, 9) using `refs/qc.py` as the shared physics engine. The experiment-facing data loaders (`PAULA_V`, HDF5 exporters) from the original lab notebooks are stripped out.

## What the paper computes

A single ion's motional mode (frequency $\omega_1 \sim 2\pi \cdot 2.8\,\mathrm{MHz}$) is prepared in

$$ |\psi_0\rangle = \hat{D}(\alpha)\,\hat{S}(\zeta)\,|0\rangle,\qquad \hat{D}(\alpha)=e^{\alpha a^\dagger - \alpha^* a},\quad \hat{S}(\zeta)=e^{(\zeta^* a^2 - \zeta a^{\dagger 2})/2}. $$

A carrier-like drive with the **full Lamb-Dicke operator** is applied, amplitude-modulated by a stroboscopic or sinusoidal waveform at $n\omega_1$:

$$ \hat{H}(t) = \tfrac{\omega_z}{2}\sigma_z + \omega_1 a^\dagger a + m(t)\cdot\tfrac{\Omega}{2}\left(\sigma_+\, e^{i\eta(a+a^\dagger)} + \mathrm{h.c.}\right).$$

$\eta$ is the Lamb-Dicke parameter, $m(t)$ is the AOM-filtered strobe envelope (mode `mod_type=0`) or a pure sinusoid (`mod_type=1`). Readout returns $\langle\sigma_{x,y,z}\rangle$ and $\delta\langle n\rangle$ as a function of the state's displacement / squeeze parameters.

## Figure → code map

| Paper fig | Physics | Scan | Section below |
|-----------|---------|------|---------------|
| Fig. 3    | Displaced state probe, $\langle\sigma_z\rangle$ vs $\psi$ | 1D, phase of $\alpha$ | §3 |
| Fig. 6    | Back-action on displaced states | 2D, $(\|\alpha\|,\varphi_0)$ | §4 |
| Fig. 8    | $\varphi_0,C \to \langle x\rangle,\langle p\rangle$ calibration | derived from §3 | §5 |
| Fig. 9    | Squeezed-vacuum tomography | 1D + 2D $(\|\zeta\|,\varphi_0)$ | §6 |

## §1 — Setup

Imports `QC` from `refs/qc.py`. Requires `qutip`, `adaptive`, `scipy`, `numpy`, `matplotlib`.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

import os, sys, numpy as np, matplotlib.pyplot as plt
import qutip
import scipy.constants as cst
from scipy.signal import butter, filtfilt, square
from scipy.interpolate import CubicSpline

# Make refs/qc.py importable regardless of where the notebook is run from.
REFS = os.path.abspath(os.path.join(os.getcwd(), '..', 'refs'))
if REFS not in sys.path:
    sys.path.insert(0, REFS)
from qc import QC
qc = QC(dark=False)

import adaptive
from adaptive.runner import simple as run_sync
from adaptive.learner.learner1D import curvature_loss_function
curvature_loss = curvature_loss_function()

## §2 — Physics primer & core solver

### Lamb-Dicke parameter
For a laser k-vector $\mathbf{k}$ projected onto a motional mode direction $\hat{m}$,
$$ \eta = (\mathbf{k}\cdot\hat{m})\sqrt{\hbar/(2 M \omega_1)}. $$
`qc.lamb_dicke(kvec, mvec, omega_MHz, mass)` returns this directly for $^{25}\mathrm{Mg}^{+}$. In the paper geometry $\mathbf{k}\propto(0,-1,1)/\sqrt2$ and the mode is along $\hat{z}$ (set `theta_m=0`).

### Fock-space truncation
`qc.initialise_single_mode` auto-selects `Ncut` so population above the cutoff is below `Prec` (default $10^{-10}$). For $\|\alpha\|\lesssim 5$ or $\|\zeta\|\lesssim 1.8$, `Ncut ≈ 2|α|² + 10` or `Ncut ≈ 50` is plenty. We expose it as an argument for convergence checks.

### Stroboscopic envelope
The drive is a duty-cycle square wave at $m\omega_1$ (`mod_fac=m`, typically 1 or 2), passed through a Butterworth low-pass mimicking the finite AOM sound-velocity response (erf-like rise). `sel='fast'` uses a $4\times$ higher cutoff than `sel='slow'` (different AOM geometries in the paper).

### Pulse-area / duration
With the drive gated on for a small duty-cycle $d$, an effective $\pi/2$ rotation takes $\tau \approx (2\pi/\Omega_\mathrm{eff})/(4d)$, where $\Omega_\mathrm{eff}=\Omega e^{-\eta^2/2}$ is the Debye-Waller-corrected carrier Rabi.

In [ ]:
def simulate_stroboscopic(
    sel='fast',
    spin_ini=(0, 0),           # (theta/2pi, extra phase/2pi) for single spin
    mode_para=(0.001, (0, 0), (0, 0), 2.8),
    # mode_para = (n_thermal, (|alpha|, phase_alpha/2pi), (|zeta|, phase_zeta/2pi), freq_MHz)
    theta_m=0.0,               # mode-axis tilt (deg)
    drive_para=(0.25, 0.0),    # (Omega/(2pi MHz), detuning/omega_1)
    mod_on=True, mod_type=0,   # 0 = stroboscopic, 1 = sinusoidal
    mod_fac=1, strobo_dur=0.025,
    Ncut=35, do_plot=False,
):
    """Spin + motion evolution under AC-modulated Lamb-Dicke carrier.

    Returns (output, prop_l_S, contrast) where
      - output        : qutip.solver.Result (full state history)
      - prop_l_S[t,:] = [<sigma_x>, <sigma_y>, <sigma_z>, entropy]
      - contrast      = (max(sigma_z) - min(sigma_z)) / 2
    """
    # ----- unpack parameters -----
    n_th, (dis_a, dis_phs_2pi), (sq_a, sq_phs_2pi), freq_MHz = mode_para
    omega_1 = 2 * np.pi * freq_MHz
    omega_z = drive_para[1] * omega_1
    Omega   = 2 * np.pi * drive_para[0]
    dis_phs = dis_phs_2pi * 2 * np.pi
    sq_phs  = sq_phs_2pi * 2 * np.pi
    s_ini, s_phs = spin_ini
    r_spin = [s_ini * 2 * np.pi, 0.0, (0.677/2 + s_phs) * 2 * np.pi]

    # ----- Lamb-Dicke (paper k-geometry) -----
    deg = np.pi / 180
    kvec = [0, -np.sqrt(2)/np.sqrt(2), np.sqrt(2)/np.sqrt(2)]
    mvec = [0, -np.sin(theta_m * deg), np.cos(theta_m * deg)]
    eta_1 = qc.lamb_dicke(kvec, mvec, omega_1 * cst.mega, 25 * cst.atomic_mass)
    Omega_eff = np.exp(-eta_1**2 / 2) * Omega

    # ----- initial state -----
    rho_spin_0, _ = qc.initialise_spins(no=1, angle=r_spin, verbose=do_plot)
    rho_motion_0, props_m = qc.initialise_single_mode(
        n_th=n_th, Fck=0, sq_ampl=sq_a, sq_phi=sq_phs,
        dis_ampl=dis_a, dis_phi=dis_phs,
        Prec=1e-10, Ncut=Ncut, verbose=do_plot,
    )
    N = props_m[0]
    rho_0 = qutip.tensor(rho_motion_0, rho_spin_0)

    # ----- operators & Hamiltonian -----
    a  = qutip.tensor(qutip.destroy(N), qutip.qeye(2))
    sm = qutip.tensor(qutip.qeye(N), qutip.destroy(2))
    sz = qutip.tensor(qutip.qeye(N), qutip.sigmaz())

    H0 = omega_z/2 * sz + omega_1 * a.dag() * a
    C  = (1j * eta_1 * (a.dag() + a) + 1j).expm()       # full LD carrier
    HI = Omega/2 * (sm.dag() * C + sm * C.dag())

    # ----- drive envelope m(t) -----
    if mod_on:
        omega_mod   = mod_fac * omega_1
        duty_cycle  = strobo_dur / (2*np.pi / omega_mod)
        t_end = (0.98 if sel=='fast' else 0.9) * (2*np.pi/Omega_eff) / 4 / duty_cycle
        if mod_type == 1:
            t_end = 4.2
        tlist = np.linspace(0, t_end, int(t_end * 300))

        def mod_fct(t):
            return (square(omega_mod * (t - strobo_dur), duty=duty_cycle) + 1) / 2

        def aom_filter(data, cutoff, fs, order=1):
            b, a_c = butter(order, cutoff / 3.0, btype='low', analog=False)
            return filtfilt(b, a_c, data)

        f_cutoff = 4 * 0.2 if sel == 'fast' else 0.2
        fs = 1 / (t_end / len(tlist))
        mod_sampled = np.abs(aom_filter(mod_fct(tlist), f_cutoff, fs))
        mod_spline = CubicSpline(tlist, mod_sampled)

        def mod(t, args):
            if mod_type == 0:
                return mod_spline(t)
            f_mod = mod_fac * omega_1 / (2 * np.pi)
            _, _, depth = qc.UV_AOM_Specs(1, f_mod, verbose=False)
            return qc.generate_sinusoidal(t, f_mod, depth)

        H = [H0, [HI, mod]]
    else:
        t_end = (2*np.pi / Omega_eff) / 4
        tlist = np.linspace(0, t_end, int(t_end * 200))
        H = H0 + HI

    # ----- solve & reduce -----
    output = qutip.mesolve(H, rho_0, tlist, [], [], progress_bar=None)
    prop_l_S, _ = qc.trace_spin_props(output, ptrace_sel=[1], verbose=False)
    sz_l = prop_l_S[:, 2]
    contrast = (np.max(sz_l) - np.min(sz_l)) / 2

    if do_plot:
        print(f'eta = {eta_1:.3f}   Omega_eff/(2π) = {Omega_eff/(2*np.pi):.4f} MHz   t_end = {t_end:.3f} µs')
        fig, ax = plt.subplots(figsize=(5, 2.5))
        ax.plot(tlist, (prop_l_S[:, 2]+1)/2, label=r'$P_\downarrow$')
        ax.set_xlabel(r'$t$ (µs)'); ax.set_ylabel('population'); ax.legend(); ax.grid(True)
        plt.show()

    return output, prop_l_S, contrast

### Sanity check: pulse-shape + single-trace dynamics

In [ ]:
_ = simulate_stroboscopic(
    sel='fast',
    spin_ini=(0.25, 0),
    mode_para=(0.001, (3.4, 0.25), (0, 0), 2.8),
    drive_para=(0.25, 0),
    mod_on=True, mod_type=0, mod_fac=1, strobo_dur=0.025,
    Ncut=35, do_plot=True,
)

## §3 — Fig. 3: Coherent displaced state, 1D scan in $\psi$

Fix $\|\alpha\| = 3.4(1)$, sweep the phase $\psi \in [0,2\pi)$, record the four observables $(\sigma_x, \sigma_y, \sigma_z, S)$ at the end of the pulse. The adaptive 1D learner samples denser near inflection points. The contrast in $\sigma_z$ is the key figure of merit.

In [ ]:
DIS_AMP = 3.4       # |alpha|
MODE_FREQ = 2.8     # MHz
AOM = 'fast'
STROBO = 0.025 if AOM == 'fast' else 0.055

def probe_displaced_phase(phase_2pi):
    _, prop_l_S, _ = simulate_stroboscopic(
        sel=AOM,
        spin_ini=(-0.25, 0.0),
        mode_para=(0.001, (DIS_AMP, phase_2pi), (0, 0), MODE_FREQ),
        drive_para=(0.25, 0),
        mod_on=True, mod_type=0, mod_fac=1, strobo_dur=STROBO,
        Ncut=35,
    )
    return tuple(prop_l_S[-1])     # (sigma_x, sigma_y, sigma_z, entropy)

learner_fig3 = adaptive.Learner1D(probe_displaced_phase, bounds=(0.0, 1.0))
run_sync(learner_fig3, goal=lambda l: l.npoints > 32)

In [ ]:
print(f'Fig. 3 scan: {learner_fig3.npoints} points')

In [ ]:
def plot_1d_scan(learner, xlabel, title):
    xs = np.array(sorted(learner.data.keys()))
    ys = np.array([learner.data[x] for x in xs])
    sx, sy, sz, S = ys.T
    cx, cy, cz = [(v.max() - v.min())/2 for v in (sx, sy, sz)]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 4), sharex=True,
                                   gridspec_kw={'height_ratios':[3,1]})
    ax1.set_title(title)
    ax1.plot(xs, sx, '-o', color='red',    label=fr'$\sigma_x$  (C={cx:.2f})')
    ax1.plot(xs, sy, '-o', color='navy',   label=fr'$\sigma_y$  (C={cy:.2f})')
    ax1.plot(xs, sz, '-o', color='orange', label=fr'$\sigma_z$  (C={cz:.2f})')
    ax1.plot(xs, np.sqrt(sx**2+sy**2+sz**2), '-o', color='black', label='coherence')
    ax1.set_ylim(-1.05, 1.05); ax1.set_ylabel(r'$\langle\sigma\rangle$')
    ax1.legend(loc='lower right', fontsize=8); ax1.grid(True)
    ax2.plot(xs, S, '-o', color='black')
    ax2.set_xlabel(xlabel); ax2.set_ylabel('entropy'); ax2.grid(True)
    plt.tight_layout(); plt.show()

plot_1d_scan(learner_fig3, r'phase of $\alpha$ ($2\pi$)',
             fr'Fig. 3 — $|\alpha|={DIS_AMP}$, $\omega_1/2\pi={MODE_FREQ}$ MHz, AOM: {AOM}')

## §4 — Fig. 6: 2D back-action map for displaced states

Scan jointly over $(\|\alpha\|,\varphi_0)$. For each point we record $\langle\sigma_z\rangle$ (measurement signal) and $\delta\langle n\rangle = \langle n\rangle_\mathrm{fin} - \langle n\rangle_\mathrm{ini}$ (back-action). The 2D adaptive learner with curvature loss concentrates samples on fringe boundaries.

> **Convergence note.** For $\|\alpha\|\lesssim 4.5$, `Ncut=35` is sufficient. Increase to 50–60 if extending to $\|\alpha\|>5$.

In [ ]:
def probe_displaced_2d(xy):
    amp, phs_2pi = xy
    out, prop_l_S, _ = simulate_stroboscopic(
        sel=AOM,
        spin_ini=(-0.25, 0.0),
        mode_para=(0.001, (amp, phs_2pi), (0, 0), MODE_FREQ),
        drive_para=(0.25, 0),
        mod_on=True, mod_type=0, mod_fac=1, strobo_dur=STROBO,
        Ncut=35,
    )
    prop_l_M, _ = qc.trace_motional_props(out, ptrace_sel=[0], verbose=False)
    delta_n = prop_l_M[-1, 1] - prop_l_M[0, 1]
    sx, sy, sz, S = prop_l_S[-1]
    return (float(sx), float(sy), float(sz), float(delta_n))

learner_fig6 = adaptive.Learner2D(probe_displaced_2d, bounds=((0.0, 4.5), (0.0, 1.0)))
run_sync(learner_fig6, goal=lambda l: l.npoints >= 60)

In [ ]:
print(f'Fig. 6 scan: {learner_fig6.npoints} points')

In [ ]:
from scipy.interpolate import griddata
from matplotlib.colors import Normalize

def plot_2d_map(learner, ch, label, title,
                xbounds, ybounds, xlabel, ylabel,
                cmap='seismic', vmin=-1, vmax=1):
    xs, ys = zip(*learner.data.keys())
    zs = np.array(list(learner.data.values()))[:, ch]
    gx, gy = np.mgrid[xbounds[0]:xbounds[1]:200j, ybounds[0]:ybounds[1]:200j]
    gz = griddata((xs, ys), zs, (gx, gy), method='linear')
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    c = ax.contourf(gx, gy, gz, 50, cmap=cmap, norm=Normalize(vmin=vmin, vmax=vmax))
    plt.colorbar(c, ax=ax, label=label)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    plt.tight_layout(); plt.show()

plot_2d_map(learner_fig6, ch=2, label=r'$\langle\sigma_z\rangle$',
            title='Fig. 6 — signal',
            xbounds=(0, 4.5), ybounds=(0, 1),
            xlabel=r'$|\alpha|$', ylabel=r'phase $\varphi_0$ ($2\pi$)')

plot_2d_map(learner_fig6, ch=3, label=r'$\delta\langle n\rangle$',
            title='Fig. 6 — back-action',
            xbounds=(0, 4.5), ybounds=(0, 1),
            xlabel=r'$|\alpha|$', ylabel=r'phase $\varphi_0$ ($2\pi$)',
            cmap='magma',
            vmin=0, vmax=None)

## §5 — Fig. 8: phase-shift / contrast → $\langle x\rangle, \langle p\rangle$ calibration

The calibration of the paper expresses the stroboscopic response in terms of a linear phase shift $\varphi_0$ (encoding $\langle x\rangle$) and a contrast reduction $C$ (encoding $\langle p\rangle$). Operationally, for a thermal probe $\bar n = 0.15$ one fits

$$ \langle\sigma_z\rangle(\varphi) = C\,\sin(\varphi - \varphi_0) $$

at three offsets $(0°, \pm 5°, \pm 90°)$ and extracts $(\varphi_0, C)$ by sinusoidal fit, then maps them onto $(\langle x\rangle, \langle p\rangle)$ via the parabolic calibration curves of Fig. 8. Below we reuse §3's scan data and illustrate the fit; a full calibration requires running §3 for a grid of $(\langle x\rangle, \langle p\rangle)$ target states.

In [ ]:
from scipy.optimize import curve_fit

def sin_model(phi, C, phi0, off):
    return C * np.sin(2*np.pi*(phi - phi0)) + off

xs = np.array(sorted(learner_fig3.data.keys()))
ys = np.array([learner_fig3.data[x] for x in xs])
sz = ys[:, 2]
p0 = [(sz.max()-sz.min())/2, xs[np.argmax(sz)] - 0.25, sz.mean()]
popt, _ = curve_fit(sin_model, xs, sz, p0=p0)
C_fit, phi0_fit, off_fit = popt
print(f'contrast C   = {C_fit:+.3f}')
print(f'phase   phi0 = {phi0_fit:+.3f}  (units of 2π)')
print(f'offset       = {off_fit:+.3f}')

phi_dense = np.linspace(0, 1, 400)
plt.figure(figsize=(5,3))
plt.plot(xs, sz, 'o', color='orange', label=r'$\langle\sigma_z\rangle$ data')
plt.plot(phi_dense, sin_model(phi_dense, *popt), '-', color='black', label='fit')
plt.xlabel(r'phase ($2\pi$)'); plt.ylabel(r'$\langle\sigma_z\rangle$')
plt.title('Fig. 8 — (C, φ₀) extraction from §3 scan')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

## §6 — Fig. 9: Squeezed-vacuum tomography

For squeezed-state probing the drive is modulated at $\mathbf{2\omega_1}$ (`mod_fac=2`, slow AOM for 1D, fast AOM for 2D). Scan first over the squeeze **phase** at fixed $\|\zeta\|=1.55$, then over the full $(\|\zeta\|,\varphi_0)$ plane. Squeezing pushes phonon occupation high, so `Ncut=51–57`.

### 6.1 — 1D phase scan (slow AOM)

In [ ]:
SQ_AMP = 1.55
AOM_SQ = 'slow'
STROBO_SQ = 0.055 if AOM_SQ == 'slow' else 0.025

def probe_squeezed_phase(phase_2pi):
    _, prop_l_S, _ = simulate_stroboscopic(
        sel=AOM_SQ,
        spin_ini=(0.25, 0.0),
        mode_para=(0.001, (0, 0), (SQ_AMP, phase_2pi), MODE_FREQ),
        drive_para=(0.25, 0),
        mod_on=True, mod_type=0, mod_fac=2, strobo_dur=STROBO_SQ,
        Ncut=51,
    )
    return tuple(prop_l_S[-1])

learner_fig9a = adaptive.Learner1D(probe_squeezed_phase, bounds=(0.0, 1.0))
run_sync(learner_fig9a, goal=lambda l: l.npoints > 20)

In [ ]:
print(f'Fig. 9a scan: {learner_fig9a.npoints} points')

In [ ]:
plot_1d_scan(learner_fig9a, r'squeeze phase ($\pi$)',
             fr'Fig. 9a — $|\zeta|={SQ_AMP}$, AOM: {AOM_SQ}')

### 6.2 — 2D $(|\zeta|,\varphi_0)$ map (fast AOM)

In [ ]:
AOM_SQ2D = 'fast'
STROBO_SQ2D = 0.025

def probe_squeezed_2d(xy):
    amp, phs_2pi = xy
    _, prop_l_S, _ = simulate_stroboscopic(
        sel=AOM_SQ2D,
        spin_ini=(0.25, 0.0),
        mode_para=(0.001, (0, 0), (amp, phs_2pi), MODE_FREQ),
        drive_para=(0.25, 0),
        mod_on=True, mod_type=0, mod_fac=2, strobo_dur=STROBO_SQ2D,
        Ncut=41,       # reduced from 57 — still OK for |zeta|<=1.8 (pop<1e-5 at n=41)
    )
    return tuple(float(v) for v in prop_l_S[-1])

learner_fig9b = adaptive.Learner2D(probe_squeezed_2d, bounds=((0.02, 1.6), (0.0, 0.5)))
run_sync(learner_fig9b, goal=lambda l: l.npoints >= 50)

In [ ]:
print(f'Fig. 9b scan: {learner_fig9b.npoints} points')

In [ ]:
for ch, lab in [(0, r'$\sigma_x$'), (1, r'$\sigma_y$'), (2, r'$\sigma_z$')]:
    plot_2d_map(learner_fig9b, ch=ch, label=lab,
                title=f'Fig. 9b — {lab}',
                xbounds=(0.02, 1.8), ybounds=(0.0, 0.5),
                xlabel=r'amplitude $r$', ylabel=r'phase $\varphi$ ($\pi$)')

## Reproduction checklist

- [x] Core solver (`simulate_stroboscopic`) — full LD carrier, AOM-filtered strobe, QuTiP `mesolve`
- [x] Fig. 3 — 1D $\psi$ scan of $D(\alpha)|0\rangle$
- [x] Fig. 6 — 2D $(|\alpha|,\varphi_0)$ back-action map
- [x] Fig. 8 — $(C,\varphi_0)$ extraction by sinusoidal fit
- [x] Fig. 9 — squeezed-vacuum 1D phase + 2D $(|\zeta|,\varphi_0)$

**Parameters not yet validated against the paper's exact numbers** (all of them are tunable in the cells above): $\Omega/(2\pi) = 0.25$ MHz, $\bar n = 0.001$, $\omega_1/(2\pi) = 2.8$ MHz, $\eta$ derived from a $45°$ Raman geometry. The paper quotes $\Omega/(2\pi)\approx 0.15$ MHz and $\bar n \approx 0.15$ for Fig. 8 — switch those in `mode_para[0]` and `drive_para[0]` to match published curves exactly.

**Gap:** Fig. 2 (polarization-gradient AC-Stark pattern) is not reproduced here — it needs a separate Stark-shift routine not present in `qc.py`.